# Quickstart

## 1. Load the LongMemEval dataset from the Hub

In [1]:
import os
from dotenv import find_dotenv, load_dotenv

# walks up from the cwd until it finds a .env
load_dotenv(find_dotenv(usecwd=True))
HF_TOKEN = os.environ["HUGGINGFACE_TOKEN"]

In [2]:
from huggingface_hub import hf_hub_download

DATA_PATH = hf_hub_download(
    repo_id="xiaowu0162/longmemeval-cleaned",
    filename="longmemeval_oracle.json",
    repo_type="dataset",
    token=HF_TOKEN,
)

DATA_PATH

/home/ssubrahmanya/summary-mem/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'/home/ssubrahmanya/.cache/huggingface/hub/datasets--xiaowu0162--longmemeval-cleaned/snapshots/98d7416c24c778c2fee6e6f3006e7a073259d48f/longmemeval_oracle.json'

In [3]:
import json

dataset = json.loads(open(DATA_PATH).read())
len(dataset)

500

## 2. Pick a random sample & describe it

In [4]:
import random

sample = random.choice(dataset)
sample["question_id"]

'945e3d21'

In [5]:
sessions = sample["haystack_sessions"]
turns_per_session = [len(s) for s in sessions]

print("question_id     :", sample["question_id"])
print("question_type   :", sample["question_type"])
print("question_date   :", sample.get("question_date"))
print("num_sessions    :", len(sessions))
print("turns/session   :", turns_per_session)
print("total_turns     :", sum(turns_per_session))
print("min/avg/max     :", min(turns_per_session), sum(turns_per_session) / len(sessions), max(turns_per_session))
print("question        :", sample["question"])
print("gold answer     :", sample["answer"])

question_id     : 945e3d21
question_type   : knowledge-update
question_date   : 2023/12/25 (Mon) 05:30
num_sessions    : 2
turns/session   : [12, 12]
total_turns     : 24
min/avg/max     : 12 12.0 12
question        : How often do I attend yoga classes to help with my anxiety?
gold answer     : Three times a week.


## 3. Run summary-mem on the sample

### 3.1 Load & instantiate

In [6]:
from summary_mem.clients import get_chat_client
from summary_mem.eval import MemoryEvaluator

client = get_chat_client()
evaluator = MemoryEvaluator(
    client,
    conversation_id=sample["question_id"],
    db_path="sample_memory.db",
)

### 3.2 Index the sessions into memory

In [7]:
evaluator.index(sample["haystack_sessions"], sample.get("haystack_dates"))
evaluator.memory.recall(sample["question_id"])

{'assistant': "The assistant is designed to help users prioritize their work tasks and support them in managing stress and mental health. They offer guidance on task prioritization using the Eisenhower Matrix, assisting users in categorizing tasks based on urgency and importance. The assistant encourages users to share their tasks and provides recommendations for handling them effectively, including the creation of detailed schedules.\n\nThe assistant emphasizes the importance of self-care and mindfulness practices. They suggest various stress-reducing apps and meditation techniques, such as mindfulness meditation, loving-kindness meditation, body scan, and 4-7-8 breathing, to help users manage anxiety and improve focus. The assistant also provides journaling prompts and reflection exercises for users exploring childhood trauma and emotional healing, stressing the significance of self-compassion, boundary setting, and seeking professional help.\n\nTo enhance users' understanding of anx

### 3.3 Query with the sample's question

In [8]:
result = evaluator.rag_qa(
    sample["question"],
    question_date=sample.get("question_date"),
    gold_answer=sample["answer"],
)

### 3.4 Compare against the gold answer

In [9]:
print("Q     :", result.question)
print("GOLD  :", result.gold)
print("MODEL :", result.answer)
print("CORRECT:", result.correct)

evaluator.close()

Q     : How often do I attend yoga classes to help with my anxiety?
GOLD  : Three times a week.
MODEL : You practice yoga three times a week to help clear your head and improve focus.
CORRECT: True
